# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Ovarian's disease *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [9]:
# TODO 3.1 — load the three ovarian cancer matrices and inspect them.

tx = pd.read_csv(
    "raw_ov/OV_RNAseq_gene_RSEM_coding_UQ_1500_log2_Tumor.txt",
    sep="\t",
    index_col=0
)

pr = pd.read_csv(
    "raw_ov/OV_proteomics_gene_abundance_log2_reference_intensity_normalized_Tumor.txt",
    sep="\t",
    index_col=0
)

gw = pd.read_csv(
    "raw_ov/OV_somatic_mutation_gene_level_binary.txt",
    sep="\t",
    index_col=0
)

print("RNA shape:", tx.shape)
print("Protein shape:", pr.shape)
print("Mutation shape:", gw.shape)

print("\nRNA index:", tx.index[:5].tolist())
print("Protein index:", pr.index[:5].tolist())
print("Mutation index:", gw.index[:5].tolist())

print("\nRNA columns:", tx.columns[:5].tolist())
print("Protein columns:", pr.columns[:5].tolist())
print("Mutation columns:", gw.columns[:5].tolist())

RNA shape: (60669, 82)
Protein shape: (10589, 83)
Mutation shape: (8298, 82)

RNA index: ['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.12', 'ENSG00000000457.14', 'ENSG00000000460.17']
Protein index: ['ENSG00000000003.15', 'ENSG00000000419.12', 'ENSG00000000457.14', 'ENSG00000000460.17', 'ENSG00000000938.13']
Mutation index: ['ENSG00000000971.16', 'ENSG00000001036.14', 'ENSG00000001084.13', 'ENSG00000001461.17', 'ENSG00000001561.7']

RNA columns: ['01OV007', '01OV017', '01OV018', '01OV023', '01OV026']
Protein columns: ['02OV032', '04OV057', '04OV028', '15OV001', '17OV011']
Mutation columns: ['01OV007', '01OV017', '01OV018', '01OV023', '01OV026']


In [10]:
# Check overlap before harmonization

common_samples = tx.columns.intersection(pr.columns).intersection(gw.columns)
print("Shared samples across all 3 layers:", len(common_samples))

common_genes_raw = tx.index.intersection(pr.index).intersection(gw.index)
print("Shared genes before ID cleanup:", len(common_genes_raw))

Shared samples across all 3 layers: 82
Shared genes before ID cleanup: 4653


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [11]:
# Clean Ensembl gene IDs by removing version suffixes

tx.index = tx.index.str.split(".").str[0]
pr.index = pr.index.str.split(".").str[0]
gw.index = gw.index.str.split(".").str[0]

print("RNA duplicate IDs:", tx.index.duplicated().sum())
print("Protein duplicate IDs:", pr.index.duplicated().sum())
print("Mutation duplicate IDs:", gw.index.duplicated().sum())

common_genes = tx.index.intersection(pr.index).intersection(gw.index)
print("Shared genes after ID cleanup:", len(common_genes))

RNA duplicate IDs: 45
Protein duplicate IDs: 0
Mutation duplicate IDs: 0
Shared genes after ID cleanup: 4653


In [12]:
# Collapse duplicated RNA Ensembl IDs by mean expression
tx = tx.groupby(tx.index).mean()

print("RNA shape after collapsing duplicates:", tx.shape)
print("RNA duplicate IDs after collapse:", tx.index.duplicated().sum())

RNA shape after collapsing duplicates: (60624, 82)
RNA duplicate IDs after collapse: 0


In [13]:
common_genes = tx.index.intersection(pr.index).intersection(gw.index)
print("Shared genes after duplicate cleanup:", len(common_genes))

Shared genes after duplicate cleanup: 4653


In [14]:
# 3.2 — harmonize gene identifiers

# Remove Ensembl version suffixes, e.g. ENSG00000141510.18 -> ENSG00000141510
tx.index = tx.index.str.split(".").str[0]
pr.index = pr.index.str.split(".").str[0]
gw.index = gw.index.str.split(".").str[0]

# Collapse duplicate RNA Ensembl IDs by mean expression
tx = tx.groupby(tx.index).mean()

print("RNA duplicate IDs:", tx.index.duplicated().sum())
print("Protein duplicate IDs:", pr.index.duplicated().sum())
print("Mutation duplicate IDs:", gw.index.duplicated().sum())

common_genes = tx.index.intersection(pr.index).intersection(gw.index)

print("Shared genes after cleanup:", len(common_genes))
print("Example IDs:", common_genes[:10].tolist())

RNA duplicate IDs: 0
Protein duplicate IDs: 0
Mutation duplicate IDs: 0
Shared genes after cleanup: 4653
Example IDs: ['ENSG00000000971', 'ENSG00000001036', 'ENSG00000001561', 'ENSG00000001626', 'ENSG00000002549', 'ENSG00000002726', 'ENSG00000003147', 'ENSG00000003393', 'ENSG00000003402', 'ENSG00000003436']


In [15]:
import mygene

mg = mygene.MyGeneInfo()

results = mg.querymany(
    common_genes.tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

id_to_symbol = {}

for r in results:
    if "symbol" in r and not r.get("notfound", False):
        id_to_symbol[r["query"]] = r["symbol"]

print("Ensembl IDs queried:", len(common_genes))
print("Mapped to gene symbols:", len(id_to_symbol))

1 input query terms found no hit:	['ENSG00000130723']


Ensembl IDs queried: 4653
Mapped to gene symbols: 4652


In [16]:
# Keep only Ensembl IDs that successfully mapped to gene symbols
mapped_ids = list(id_to_symbol.keys())

tx2 = tx.loc[mapped_ids].copy()
pr2 = pr.loc[mapped_ids].copy()
gw2 = gw.loc[mapped_ids].copy()

# Replace Ensembl IDs with gene symbols
tx2.index = tx2.index.map(id_to_symbol)
pr2.index = pr2.index.map(id_to_symbol)
gw2.index = gw2.index.map(id_to_symbol)

print("RNA shape:", tx2.shape)
print("Protein shape:", pr2.shape)
print("Mutation shape:", gw2.shape)

print("RNA duplicate gene symbols:", tx2.index.duplicated().sum())
print("Protein duplicate gene symbols:", pr2.index.duplicated().sum())
print("Mutation duplicate gene symbols:", gw2.index.duplicated().sum())

print("\nExample gene symbols:", tx2.index[:10].tolist())

RNA shape: (4652, 82)
Protein shape: (4652, 83)
Mutation shape: (4652, 82)
RNA duplicate gene symbols: 0
Protein duplicate gene symbols: 0
Mutation duplicate gene symbols: 0

Example gene symbols: ['CFH', 'FUCA2', 'ENPP4', 'CFTR', 'LAP3', 'AOC1', 'ICA1', 'ALS2', 'CFLAR', 'TFPI']


In [17]:
# Align all three layers to the same tumor samples

common_samples = (
    tx2.columns
    .intersection(pr2.columns)
    .intersection(gw2.columns)
)

tx2 = tx2.loc[:, common_samples]
pr2 = pr2.loc[:, common_samples]
gw2 = gw2.loc[:, common_samples]

print("Matched samples:", len(common_samples))
print("RNA:", tx2.shape)
print("Protein:", pr2.shape)
print("Mutation:", gw2.shape)

Matched samples: 82
RNA: (4652, 82)
Protein: (4652, 82)
Mutation: (4652, 82)


In [18]:
# Build one gene-level summary table

rna_mean = tx2.mean(axis=1)
prot_mean = pr2.mean(axis=1)
mut_freq = gw2.mean(axis=1)

df = pd.DataFrame({
    "rna_mean": rna_mean,
    "prot_mean": prot_mean,
    "mutation_freq": mut_freq
})

print("Genes in joined table:", len(df))
display(df.head())

Genes in joined table: 4652


,rna_mean,prot_mean,mutation_freq
idx,,,
CFH,12.159810,28.369678,0.012195
FUCA2,10.720340,24.445354,0.012195
ENPP4,10.409950,24.022693,0.012195
CFTR,6.895770,18.084474,0.024390
LAP3,12.036363,28.510222,0.012195


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [19]:
from scipy import stats
import numpy as np

def safe_spearman(x, y):
    mask = x.notna() & y.notna()
    x2 = x[mask]
    y2 = y[mask]

    # need enough observations
    if len(x2) < 3:
        return np.nan

    # correlation undefined if one side is constant
    if x2.nunique() < 2 or y2.nunique() < 2:
        return np.nan

    return stats.spearmanr(x2, y2).statistic

rna_prot_corr = pd.Series({
    gene: safe_spearman(tx2.loc[gene], pr2.loc[gene])
    for gene in df.index
}, name="rna_prot_corr")

df["rna_prot_corr"] = rna_prot_corr

print("Median RNA-protein correlation:", round(df["rna_prot_corr"].median(), 3))
print("Min correlation:", round(df["rna_prot_corr"].min(), 3))
print("Max correlation:", round(df["rna_prot_corr"].max(), 3))
print("Genes with undefined correlation:", df["rna_prot_corr"].isna().sum())

display(df.head())

Median RNA-protein correlation: 0.393
Min correlation: -0.9
Max correlation: 1.0
Genes with undefined correlation: 8


,rna_mean,prot_mean,mutation_freq,rna_prot_corr
idx,,,,
CFH,12.159810,28.369678,0.012195,0.370459
FUCA2,10.720340,24.445354,0.012195,0.432392
ENPP4,10.409950,24.022693,0.012195,0.443545
CFTR,6.895770,18.084474,0.024390,0.312308
LAP3,12.036363,28.510222,0.012195,0.741188


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [20]:
# 3.4 — multi-evidence score

df["transcriptomic"] = df["rna_mean"].rank(pct=True)
df["proteomic"] = df["prot_mean"].rank(pct=True)
df["genomic"] = df["mutation_freq"].rank(pct=True)

df["score"] = multi_evidence_score(
    df,
    ["transcriptomic", "proteomic", "genomic"],
    EQUAL_WEIGHTS
)

ranked_ov = df.sort_values("score", ascending=False)

display(ranked_ov.head(15))

,rna_mean,prot_mean,mutation_freq,rna_prot_corr,transcriptomic,proteomic,genomic,score
idx,,,,,,,,
AHNAK,14.632023,30.600042,0.073171,0.140673,0.992906,0.997420,0.996991,0.995772
COL3A1,15.494590,29.764645,0.036585,0.611541,0.997420,0.992691,0.944540,0.978217
FLNA,13.313395,30.230026,0.036585,0.488352,0.980224,0.995486,0.944540,0.973416
RPSA,15.582441,28.550444,0.036585,0.172746,0.998065,0.975924,0.944540,0.972843
PRKDC,12.263348,28.383211,0.073171,0.332528,0.941745,0.973345,0.996991,0.970694
VCAN,12.142939,28.562692,0.060976,0.853854,0.930353,0.976354,0.992799,0.966502
GNAS,14.474849,26.558802,0.060976,0.319186,0.992261,0.913801,0.992799,0.966287
COPA,12.534139,27.610844,0.048780,0.080486,0.957438,0.953568,0.981406,0.964137
MUC16,12.076780,27.791411,0.134146,0.458650,0.922829,0.956793,0.999570,0.959731


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [21]:
top15 = ranked_ov.head(15)

print("Top 15 OV targets:")
display(top15)

top15.to_csv("targets_ov.csv")

print("Saved: targets_ov.csv")

Top 15 OV targets:


,rna_mean,prot_mean,mutation_freq,rna_prot_corr,transcriptomic,proteomic,genomic,score
idx,,,,,,,,
AHNAK,14.632023,30.600042,0.073171,0.140673,0.992906,0.997420,0.996991,0.995772
COL3A1,15.494590,29.764645,0.036585,0.611541,0.997420,0.992691,0.944540,0.978217
FLNA,13.313395,30.230026,0.036585,0.488352,0.980224,0.995486,0.944540,0.973416
RPSA,15.582441,28.550444,0.036585,0.172746,0.998065,0.975924,0.944540,0.972843
PRKDC,12.263348,28.383211,0.073171,0.332528,0.941745,0.973345,0.996991,0.970694
VCAN,12.142939,28.562692,0.060976,0.853854,0.930353,0.976354,0.992799,0.966502
GNAS,14.474849,26.558802,0.060976,0.319186,0.992261,0.913801,0.992799,0.966287
COPA,12.534139,27.610844,0.048780,0.080486,0.957438,0.953568,0.981406,0.964137
MUC16,12.076780,27.791411,0.134146,0.458650,0.922829,0.956793,0.999570,0.959731


Saved: targets_ov.csv


In [22]:
# Inspect RNA-protein discordant genes for report interpretation

discordant = ranked_ov.sort_values("rna_prot_corr").head(10)

display(discordant[
    ["rna_mean", "prot_mean", "mutation_freq", "rna_prot_corr", "score"]
])

,rna_mean,prot_mean,mutation_freq,rna_prot_corr,score
idx,,,,,
MDM4,10.192187,17.071182,0.012195,-0.900000,0.337561
INO80D,9.864557,13.993704,0.024390,-0.761905,0.440026
LRRCC1,9.747541,21.497745,0.024390,-0.733333,0.551806
WNT5B,5.991284,16.088701,0.012195,-0.700000,0.158355
TSHZ1,9.715529,15.506014,0.012195,-0.690476,0.279808
MARK3,11.117680,16.923591,0.012195,-0.661538,0.403339
ADGRG4,0.585645,17.027551,0.048780,-0.654654,0.351999
ZKSCAN5,9.566034,15.529929,0.012195,-0.642857,0.268487
UNC45B,1.476977,22.053326,0.036585,-0.630062,0.460089


### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.